In [3]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field

# 🔑 Set your API key if it's not already in your system environment variables

os.environ["OPENAI_API_KEY"] = "<Enter your APIs>"

# 1. Initialize the OpenAI Client
client = OpenAI()

# 2. Define the exact Pydantic structural blueprint we expect from the LLM
class TestCaseLog(BaseModel):
    test_id: int = Field(description="The unique numerical identifier for the test case.")
    test_name: str = Field(description="The plain text name of the feature being verified.")
    status: str = Field(description="Must be exactly one of these strings: 'PASSED', 'FAILED', or 'SKIPPED'.")
    execution_time_ms: float = Field(description="The duration of the test execution in milliseconds.")

# 3. Simulate an unstructured human conversation/log input from a QA engine
user_prompt = (
    "Hey team, I just ran the automation suite for our login module. "
    "Test number 4092 completed. The 'User Login Authentication' flow ran successfully "
    "and took exactly 124.5 milliseconds to execute without any errors."
)

print("--- Sending Unstructured Input to LLM ---")
print(f"User Input: '{user_prompt}'\n")

# 4. Call the OpenAI API using Structured Outputs
# We pass our Pydantic model directly into the response_format parameter!
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful QA assistant that extracts structured data from text logs."},
        {"role": "user", "content": user_prompt}
    ],
    response_format=TestCaseLog # This forces the LLM to strictly follow our Pydantic schema
)

# 5. Extract the parsed data object
structured_data = response.choices[0].message.parsed

print("--- Extracted & Validated Object Received ---")
print(f"Object Type: {type(structured_data)}")
print(f"Test ID: {structured_data.test_id} (Type: {type(structured_data.test_id)})")
print(f"Test Name: {structured_data.test_name}")
print(f"Status: {structured_data.status}")
print(f"Execution Time: {structured_data.execution_time_ms} ms (Type: {type(structured_data.execution_time_ms)})")

--- Sending Unstructured Input to LLM ---
User Input: 'Hey team, I just ran the automation suite for our login module. Test number 4092 completed. The 'User Login Authentication' flow ran successfully and took exactly 124.5 milliseconds to execute without any errors.'

--- Extracted & Validated Object Received ---
Object Type: <class '__main__.TestCaseLog'>
Test ID: 4092 (Type: <class 'int'>)
Test Name: User Login Authentication
Status: PASSED
Execution Time: 124.5 ms (Type: <class 'float'>)
